# MineDB Unified Self-Bootstrapping Pipeline (`Qwen/Qwen2.5-3B-Instruct` FP16 + MotherDuck)
Fine-tuned queries & Arabic script gate with automatic run summary save & session shutdown on completion.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import asyncio
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from urllib.parse import parse_qs, unquote, urlparse, urlunparse

def log(msg: str):
    ts = datetime.now(timezone.utc).strftime("%H:%M:%S")
    print(f"[{ts} UTC] {msg}", flush=True)

log("=== PIPELINE STARTED ===")
log("[1.1] Initializing configuration & constants...")

# ----------------------------- CONFIG ---------------------------------------
DUCKDB_VERSION = "1.5.6"                  # must match MotherDuck-supported version
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"     # ~6.2GB FP16 weights; fast tuning now, swap to 14B later
DATASET_SLUG = "minedb-cache-3b-v1"       # stored as <KAGGLE_USERNAME>/minedb-cache-3b-v1
DATASET_TITLE = "MineDB Cache 3B v1"

BUFFER_PATH = "/kaggle/input/raw-scrape-buffer/raw_batch.parquet"
CACHE_TMP_ROOT = "/tmp/minedb_cache"      # uses /tmp so /kaggle/working stays clean on save
MAX_PER_QUERY = 12
FETCH_CONCURRENCY = 16
RAW_TEXT_CHARS = 1800
LLM_BATCH_SIZE = 2                        # safe batch size on 15GB T4 (avoids 1.45GB logits spike)
MAX_NEW_TOKENS = 384                      # prevents JSON truncation on longer Arabic/multi-field outputs
MIN_VALUE_SCORE = 7
RETENTION_DAYS = 60
SEARCH_DELAY_S = 1.8                      # polite delay + jitter to prevent SERP throttling
AUTO_SAVE_AND_SHUTDOWN = True             # auto-save notebook snapshot & stop Kaggle session on completion

NICHE_QUERIES = {
    "real_estate": [
        "UAE property price drop below market",
        "Dubai Abu Dhabi distressed villa apartment sale",
        "motivated seller apartment villa Dubai Abu Dhabi AED",
        "off plan property payment plan handover Dubai Abu Dhabi",
        "commercial office retail warehouse lease Dubai Abu Dhabi AED",
        "townhouse penthouse sale listing Dubai Marina Downtown Abu Dhabi",
    ],
    "b2b_leads": [
        "hiring software engineer cloud devops UAE",
        "Dubai Abu Dhabi tech startup hiring engineer",
        "UAE enterprise cloud infrastructure hiring",
        "fintech AI data engineer jobs Dubai Abu Dhabi",
        "IT sales account executive enterprise software UAE",
        "cybersecurity product manager remote hybrid Dubai Abu Dhabi",
    ],
    "ecommerce": [
        "myshopify new collection price UAE",
        "electronics clearance sale stock UAE",
        "discount electronics store Dubai Abu Dhabi AED",
        "flash sale bundle deal AED free delivery UAE",
        "home appliances furniture sale price Dubai Abu Dhabi",
        "limited stock outlet deal AED UAE store",
    ],
    "tenders": [
        "RFP tender submission deadline UAE",
        "government procurement tender notice UAE",
        "UAE tender bid submission deadline IT construction",
        "request for proposal vendor registration Dubai Abu Dhabi",
        "prequalification tender notice infrastructure energy UAE",
        "supply and installation tender closing date UAE",
    ],
    "arabic_nlp": [
        "منتدى نقاش تجارب اسعار الامارات",
        "اسئلة واجوبة تقنية اعمال الخليج",
        "تجربتي شراء عقار او عمل في الامارات",
        "رأيكم في السكن والعمل في دبي وابوظبي",
        "مقارنة اسعار الخدمات والمنتجات في الامارات",
        "نقاش حول السوق العقاري والتجاري في الخليج",
    ],
}

NICHE_SCHEMA = {
    "real_estate": '{"price": "number or string or null", "currency": "string or null", "location": "string or null", "property_type": "string or null", "urgency_signal": "string or null"}',
    "b2b_leads":   '{"company_name": "string or null", "job_title": "string or null", "location": "string or null", "tech_stack": ["string"], "contact_or_url": "string or null"}',
    "ecommerce":   '{"product_name": "string or null", "brand": "string or null", "price": "string or number or null", "discount_pct": "string or number or null", "stock_status": "string or null"}',
    "tenders":     '{"tender_title": "string or null", "issuing_org": "string or null", "submission_deadline": "string or null", "sector": "string or null"}',
    "arabic_nlp":  '{"clean_arabic_text": "string or null", "dialect": "string or null", "domain_category": "string or null"}',
}

SYSTEM_PROMPT = (
    "You are a strict commercial data quality gatekeeper. Analyze the webpage text and output ONLY a single "
    "valid JSON object (no markdown fences, no commentary) with exact keys: "
    '"is_sellable" (boolean: true ONLY if the text contains a concrete, actionable commercial item—such as a '
    "specific property listing with price/location, an active job opening with role/company, a concrete product "
    "offer with price/availability, an active tender/RFP notice with scope/org, or genuine Arabic-script "
    "user discussion/Q&A from the UAE/Gulf; strictly false for category/directory pages, login walls, cookie "
    "notices, generic SEO articles, navigation menus, or non-Arabic pages when niche is arabic_nlp), "
    '"value_score" (integer 1-10 rating commercial specificity and completeness; >=7 requires >=2 concrete '
    "non-null fields from the target schema and, for arabic_nlp, substantial Arabic script in clean_arabic_text), "
    '"extracted_data" (JSON object strictly matching the user schema; use JSON null for missing or non-matching '
    "fields—never output placeholder strings like 'Unknown', 'N/A', 'Not specified', 'None', or hallucinated facts)."
)

log(f"  -> Target Model: {MODEL_ID} | Cache Slug: {DATASET_SLUG} | Batch Size: {LLM_BATCH_SIZE} | Max Tokens: {MAX_NEW_TOKENS} | Min Score: {MIN_VALUE_SCORE}")
log(f"  -> Auto-Save & Shutdown enabled: {AUTO_SAVE_AND_SHUTDOWN}")

# ----------------------------- SECRETS & SELF-BOOTSTRAPPING CACHE -----------
log("[1.2] Loading credentials (env vars / secrets.json / Kaggle Secrets)...")
FILE_SECRETS = {}
for root, _, files in os.walk("/kaggle/input"):
    if "secrets.json" in files:
        secret_path = os.path.join(root, "secrets.json")
        with open(secret_path, "r", encoding="utf-8") as f:
            FILE_SECRETS = json.load(f)
        log(f"  -> Found mounted secrets file: {secret_path}")
        break


def get_secret_safe(name, required=True):
    val = os.environ.get(name) or FILE_SECRETS.get(name)
    source = "env/file"
    if not val:
        try:
            from kaggle_secrets import UserSecretsClient
            val = UserSecretsClient().get_secret(name)
            source = "kaggle_secrets"
        except Exception:
            val = None
    if val:
        log(f"  -> Secret '{name}' loaded OK (source={source}, len={len(val.strip())})")
        return val.strip().strip('"').strip("'")
    if required:
        raise ValueError(f"Missing required secret: '{name}'.")
    log(f"  -> Optional secret '{name}' not set.")
    return None


md_token = get_secret_safe("MOTHERDUCK_TOKEN", required=True)
if not md_token.startswith("eyJ"):
    raise ValueError(
        "MOTHERDUCK_TOKEN does not look like a JWT (expected to start with 'eyJ'). "
        "Check GitHub Actions Secret MOTHERDUCK_TOKEN."
    )
os.environ["MOTHERDUCK_TOKEN"] = md_token

kaggle_user = get_secret_safe("KAGGLE_USERNAME", required=True)
os.environ["KAGGLE_USERNAME"] = kaggle_user

kaggle_api_token = get_secret_safe("KAGGLE_API_TOKEN", required=False)
if kaggle_api_token:
    os.environ["KAGGLE_API_TOKEN"] = kaggle_api_token
else:
    os.environ["KAGGLE_KEY"] = get_secret_safe("KAGGLE_KEY", required=True)

DATASET_ID = f"{kaggle_user}/{DATASET_SLUG}"

REQUIRED_PKGS = [
    f"duckdb=={DUCKDB_VERSION}",
    "httpx",
    "trafilatura",
    "ddgs",
    "accelerate",
]


def is_valid_wheel_cache(wheels_dir: str) -> bool:
    if not wheels_dir or not os.path.isdir(wheels_dir):
        return False
    files = [f.lower() for f in os.listdir(wheels_dir) if f.endswith(".whl")]
    return any(f.startswith(f"duckdb-{DUCKDB_VERSION}") for f in files)


def is_valid_model_cache(model_dir: str) -> bool:
    if not model_dir or not os.path.isdir(model_dir):
        return False
    files = os.listdir(model_dir)
    return "config.json" in files and any(f.endswith(".safetensors") for f in files)


def find_existing_cache():
    for root_dir in ("/kaggle/input", CACHE_TMP_ROOT):
        if not os.path.exists(root_dir):
            continue
        wheels_dir, model_dir = None, None
        for r, dirs, _ in os.walk(root_dir):
            if "wheels" in dirs and wheels_dir is None:
                cand = os.path.join(r, "wheels")
                if is_valid_wheel_cache(cand):
                    wheels_dir = cand
            if "model" in dirs and model_dir is None:
                cand = os.path.join(r, "model")
                if is_valid_model_cache(cand):
                    model_dir = cand
            if wheels_dir and model_dir:
                return wheels_dir, model_dir
    return None, None


def ensure_offline_cache():
    log("[1.3] Checking local disk (/kaggle/input & /tmp/minedb_cache) for existing cache...")
    wheels_dir, model_dir = find_existing_cache()
    if wheels_dir and model_dir:
        whl_count = len([f for f in os.listdir(wheels_dir) if f.endswith(".whl")])
        shard_count = len([f for f in os.listdir(model_dir) if f.endswith(".safetensors")])
        log(f"  -> CACHE HIT! Using local cache: wheels={wheels_dir} ({whl_count} .whl) | model={model_dir} ({shard_count} shards)")
        return wheels_dir, model_dir

    log(f"  -> No local cache found. Checking remote Kaggle Dataset '{DATASET_ID}'...")
    status = subprocess.run(
        ["kaggle", "datasets", "status", DATASET_ID],
        capture_output=True,
        text=True,
    )
    if status.returncode == 0:
        status_line = status.stdout.strip().splitlines()[-1] if status.stdout.strip() else "ready"
        log(f"  -> Remote dataset '{DATASET_ID}' exists (status: {status_line}). Attempting download to {CACHE_TMP_ROOT}...")
        t0 = time.time()
        shutil.rmtree(CACHE_TMP_ROOT, ignore_errors=True)
        os.makedirs(CACHE_TMP_ROOT, exist_ok=True)
        dl = subprocess.run(
            ["kaggle", "datasets", "download", "-d", DATASET_ID, "-p", CACHE_TMP_ROOT, "--unzip", "-q"],
            capture_output=True,
            text=True,
        )
        if dl.returncode == 0:
            wheels_dir, model_dir = find_existing_cache()
            if wheels_dir and model_dir:
                log(f"  -> Downloaded & verified cache in {time.time() - t0:.1f}s.")
                return wheels_dir, model_dir
            log("  -> Dataset downloaded but valid wheels/model folders were missing; rebuilding cache...")
        else:
            err_msg = (dl.stderr or dl.stdout).strip().splitlines()[-1] if (dl.stderr or dl.stdout).strip() else "unknown error"
            log(f"  -> Remote dataset download failed ({err_msg}) — dataset shell is empty/incomplete; rebuilding & re-publishing...")

    log(f"  -> Remote dataset not found. Starting first-time bootstrap for '{DATASET_ID}'...")
    shutil.rmtree(CACHE_TMP_ROOT, ignore_errors=True)
    wheels_dir = os.path.join(CACHE_TMP_ROOT, "wheels")
    model_dir = os.path.join(CACHE_TMP_ROOT, "model")
    os.makedirs(wheels_dir, exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    t0 = time.time()
    log(f"  -> [Bootstrap 1/3] Downloading binary wheels: {REQUIRED_PKGS}...")
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "download", "-q",
            "--only-binary=:all:",
            "--dest", wheels_dir,
            *REQUIRED_PKGS,
        ],
        timeout=300,
    )
    log(f"  -> [Bootstrap 1/3] Downloaded {len(os.listdir(wheels_dir))} wheel files in {time.time() - t0:.1f}s.")

    t1 = time.time()
    log(f"  -> [Bootstrap 2/3] Downloading HuggingFace model snapshot '{MODEL_ID}'...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=model_dir,
        ignore_patterns=["*.msgpack", "*.h5", "*.ot", "*.tflite", "*.bin"],
    )
    shutil.rmtree(os.path.join(model_dir, ".cache"), ignore_errors=True)
    log(f"  -> [Bootstrap 2/3] Model snapshot saved to {model_dir} in {time.time() - t1:.1f}s.")

    t2 = time.time()
    log(f"  -> [Bootstrap 3/3] Publishing '{DATASET_ID}' to Kaggle Datasets...")
    meta = {
        "title": DATASET_TITLE,
        "id": DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }
    with open(os.path.join(CACHE_TMP_ROOT, "dataset-metadata.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    res = subprocess.run(
        ["kaggle", "datasets", "create", "-p", CACHE_TMP_ROOT, "--dir-mode", "zip"],
        capture_output=True,
        text=True,
    )
    if res.stdout.strip():
        log(f"     kaggle stdout: {res.stdout.strip()}")
    if res.stderr.strip():
        log(f"     kaggle stderr: {res.stderr.strip()}")
    if "Dataset creation error" in (res.stdout + res.stderr) or res.returncode != 0:
        log("  -> Dataset already exists or create returned non-zero; pushing new dataset version...")
        subprocess.run(
            ["kaggle", "datasets", "version", "-p", CACHE_TMP_ROOT, "-m", "Update cache", "--dir-mode", "zip"],
            check=False,
        )
    log(f"  -> [Bootstrap 3/3] Publish finished in {time.time() - t2:.1f}s.")
    return wheels_dir, model_dir


WHEELS_DIR, CACHED_MODEL_DIR = ensure_offline_cache()

log("[1.4] Installing Python packages offline from wheel cache...")
t_pip = time.time()
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-index",
        f"--find-links={WHEELS_DIR}", "--force-reinstall", "--no-deps",
        f"duckdb=={DUCKDB_VERSION}",
    ],
    timeout=120,
)
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-index",
        f"--find-links={WHEELS_DIR}",
        *[p for p in REQUIRED_PKGS if not p.startswith("duckdb==")],
    ],
    timeout=240,
)
log(f"  -> Offline pip install complete in {time.time() - t_pip:.1f}s.")

import duckdb
import httpx
import pandas as pd
import torch
import trafilatura
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS

log(f"[1.5] Runtime verified: DuckDB v{duckdb.__version__} | PyTorch {torch.__version__} | CUDA GPUs: {torch.cuda.device_count()}")
if duckdb.__version__ != DUCKDB_VERSION:
    raise RuntimeError(
        f"DuckDB v{duckdb.__version__} was already loaded in memory; click 'Run -> Restart & Clear Cell Outputs' and re-run."
    )


def run_async(coro):
    """Works both in plain scripts and inside Kaggle/Jupyter (running event loop)."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(asyncio.run, coro).result()


def save_and_shutdown_session():
    """Flush outputs, save a notebook snapshot artifact to /kaggle/working if possible, and stop the Kaggle session."""
    if not AUTO_SAVE_AND_SHUTDOWN:
        log("[7.0] AUTO_SAVE_AND_SHUTDOWN is False; leaving session running.")
        return

    log("[7.1] Saving run summary & notebook snapshot before session shutdown...")
    try:
        os.makedirs("/kaggle/working", exist_ok=True)
        summary_path = "/kaggle/working/last_run_summary.json"
        summary = {
            "completed_at_utc": datetime.now(timezone.utc).isoformat(),
            "model_id": MODEL_ID,
            "dataset_id": DATASET_ID,
            "min_value_score": MIN_VALUE_SCORE,
        }
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
        log(f"  -> Wrote run summary to {summary_path}")

        # Attempt to trigger Kaggle frontend Save Version / checkpoint if running inside IPython
        try:
            from IPython import get_ipython
            ip = get_ipython()
            if ip is not None:
                ip.run_line_magic("autosave", "1")
                log("  -> Enabled IPython autosave (1s interval).")
        except Exception as e:
            log(f"  -> IPython autosave hint skipped: {e}")

        # If running inside a Kaggle kernel container with KAGGLE_KERNEL_RUN_TYPE, try stopping via Kaggle API
        kernel_slug = os.environ.get("KAGGLE_KERNEL_RUN_TYPE")
        log(f"  -> Kernel environment detected (KAGGLE_KERNEL_RUN_TYPE={kernel_slug}).")
    except Exception as e:
        log(f"  -> Non-fatal error during save prep: {e}")

    log("[7.2] Shutting down Kaggle session now...")
    time.sleep(1.5)
    # In Kaggle notebooks, terminating the kernel process immediately stops interactive/scheduled execution
    # and frees GPU quota. Use os._exit(0) after flushing stdio so the session halts cleanly.
    sys.stdout.flush()
    sys.stderr.flush()
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is not None and getattr(ip, "kernel", None) is not None:
            ip.kernel.do_shutdown(restart=False)
    except Exception:
        pass
    os._exit(0)

In [ ]:
log(f"[2.1] Connecting to MotherDuck database 'md:minedb' (DuckDB v{duckdb.__version__})...")
t_db = time.time()
con = duckdb.connect("md:minedb")
log(f"  -> Connected to MotherDuck in {time.time() - t_db:.2f}s.")

log("[2.2] Ensuring 'seen_urls' and niche tables exist...")
con.execute("""
    CREATE TABLE IF NOT EXISTS seen_urls (
        source_url VARCHAR PRIMARY KEY,
        first_seen TIMESTAMPTZ
    )
""")
seen_total = con.execute("SELECT COUNT(*) FROM seen_urls").fetchone()[0]
log(f"  -> Table 'seen_urls' ready (currently tracking {seen_total} historical URLs).")

for _niche in NICHE_QUERIES:
    con.execute(f"""
        CREATE TABLE IF NOT EXISTS {_niche} (
            niche VARCHAR, source_url VARCHAR, scraped_at TIMESTAMPTZ,
            value_score INTEGER, cleaned_json VARCHAR
        )
    """)
    row_cnt = con.execute(f"SELECT COUNT(*) FROM {_niche}").fetchone()[0]
    log(f"  -> Table 'minedb.{_niche}' ready ({row_cnt} existing rows).")

In [ ]:
def get_con():
    global con
    try:
        con.execute("SELECT 1")
    except Exception:
        log("  -> MotherDuck connection dropped; reconnecting...")
        con = duckdb.connect("md:minedb")
    return con


# ----------------------------- URL NORMALIZATION & DISCOVERY ----------------
TRACKING_PARAMS = {
    "utm_source", "utm_medium", "utm_campaign", "utm_term", "utm_content",
    "fbclid", "gclid", "mc_cid", "mc_eid", "ref", "source",
}

SKIP_EXTENSIONS = (
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
    ".jpg", ".jpeg", ".png", ".gif", ".webp", ".svg", ".mp4", ".zip",
)


def normalize_url(raw_url: str) -> str:
    """Canonicalize URLs (strip tracking params & fragments) to maximize deduplication."""
    if not raw_url or not raw_url.startswith(("http://", "https://")):
        return ""
    try:
        p = urlparse(raw_url.strip())
        if any(p.path.lower().endswith(ext) for ext in SKIP_EXTENSIONS):
            return ""
        qs = parse_qs(p.query, keep_blank_values=False)
        clean_qs = "&".join(
            f"{k}={v[0]}" for k, v in sorted(qs.items()) if k.lower() not in TRACKING_PARAMS
        )
        clean_path = p.path.rstrip("/") or "/"
        return urlunparse((p.scheme.lower(), p.netloc.lower(), clean_path, "", clean_qs, ""))
    except Exception:
        return ""


SEARCH_BACKENDS = ["bing", "brave", "mojeek", "yahoo", "duckduckgo", "google"]


def ddg_html_fallback(query: str, max_results: int = 12) -> list[str]:
    """Direct single-request fallback against html.duckduckgo.com (no vqd token handshake)."""
    urls = []
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    try:
        with httpx.Client(headers=headers, follow_redirects=True, timeout=15.0) as client:
            resp = client.post("https://html.duckduckgo.com/html/", data={"q": query})
            if resp.status_code == 200:
                for raw_href in re.findall(r'class="result__a"[^>]*href="([^"]+)"', resp.text):
                    if "uddg=" in raw_href:
                        qs = parse_qs(urlparse(raw_href).query)
                        target = unquote(qs.get("uddg", [""])[0])
                    else:
                        target = raw_href
                    if target.startswith("http") and "duckduckgo.com" not in target:
                        urls.append(target)
                        if len(urls) >= max_results:
                            break
    except Exception:
        pass
    return urls


def discover_urls():
    log("[3.1] Running multi-backend search discovery across 5 niches...")
    found, seen = [], set()
    total_queries = sum(len(qs) for qs in NICHE_QUERIES.values())

    def add_url(niche: str, raw_url: str) -> bool:
        norm = normalize_url(raw_url)
        if norm and norm not in seen:
            seen.add(norm)
            found.append((niche, norm))
            return True
        return False

    with DDGS() as ddgs:
        q_idx = 0
        for niche, queries in NICHE_QUERIES.items():
            for q in queries:
                ordered_backends = (
                    SEARCH_BACKENDS[q_idx % len(SEARCH_BACKENDS):]
                    + SEARCH_BACKENDS[:q_idx % len(SEARCH_BACKENDS)]
                )
                q_idx += 1

                hits = []
                used_backend = "none"
                for backend in ordered_backends:
                    try:
                        res = list(
                            ddgs.text(
                                q,
                                region="wt-wt",
                                backend=backend,
                                max_results=MAX_PER_QUERY,
                            ) or []
                        )
                        if res:
                            hits = [r.get("href") or r.get("url") for r in res]
                            used_backend = backend
                            break
                    except Exception:
                        continue

                if not hits:
                    hits = ddg_html_fallback(q, max_results=MAX_PER_QUERY)
                    if hits:
                        used_backend = "html-fallback"

                added = sum(1 for href in hits if add_url(niche, href))
                log(f"  -> Query [{q_idx}/{total_queries}] [{niche}] '{q}' -> backend={used_backend}, raw_hits={len(hits)}, unique_added={added}")
                time.sleep(SEARCH_DELAY_S + random.uniform(0.2, 0.8))

    log(f"  -> Discovery complete: {len(found)} unique candidate URLs collected.")
    return found


def filter_unseen(pairs):
    log("[3.2] Filtering candidate URLs against MotherDuck 'seen_urls'...")
    db = get_con()
    existing = {r[0] for r in db.execute("SELECT source_url FROM seen_urls").fetchall()}
    unseen = [(n, u) for n, u in pairs if u not in existing]
    log(f"  -> {len(pairs)} candidates -> {len(pairs) - len(unseen)} already seen -> {len(unseen)} brand-new URLs to scrape.")
    return unseen


# ----------------------------- SCRAPING -------------------------------------
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0",
]


async def fetch_and_extract(client, sem, niche, url, stats):
    async with sem:
        for attempt in range(2):
            try:
                headers = {"User-Agent": random.choice(USER_AGENTS)}
                resp = await client.get(url, headers=headers, timeout=15.0)
                if resp.status_code in (429, 502, 503, 504) and attempt == 0:
                    await asyncio.sleep(1.5)
                    continue
                if resp.status_code >= 400:
                    stats["http_err"] += 1
                    return None
                ctype = resp.headers.get("content-type", "").lower()
                if "html" not in ctype and "text" not in ctype:
                    stats["non_html"] += 1
                    return None
                text = await asyncio.to_thread(
                    trafilatura.extract,
                    resp.text,
                    include_comments=False,
                    include_tables=True,
                    favor_precision=True,
                )
                if text:
                    clean_text = re.sub(r"\s+", " ", text).strip()
                    if len(clean_text) > 180:
                        stats["ok"] += 1
                        return {
                            "niche": niche,
                            "source_url": url,
                            "raw_text": clean_text[:RAW_TEXT_CHARS],
                            "scraped_at": datetime.now(timezone.utc),
                        }
                stats["short_or_empty"] += 1
                return None
            except Exception:
                if attempt == 0:
                    await asyncio.sleep(0.8)
        stats["timeout_or_conn"] += 1
    return None


async def scrape(targets):
    log(f"[3.3] Scraping & extracting text from {len(targets)} URLs (concurrency={FETCH_CONCURRENCY})...")
    t_scrape = time.time()
    stats = {"ok": 0, "http_err": 0, "non_html": 0, "short_or_empty": 0, "timeout_or_conn": 0}
    sem = asyncio.Semaphore(FETCH_CONCURRENCY)
    limits = httpx.Limits(max_connections=FETCH_CONCURRENCY * 2, max_keepalive_connections=FETCH_CONCURRENCY)
    async with httpx.AsyncClient(follow_redirects=True, limits=limits) as client:
        results = await asyncio.gather(*(fetch_and_extract(client, sem, n, u, stats) for n, u in targets))
    valid = [r for r in results if r]
    log(
        f"  -> Scrape finished in {time.time() - t_scrape:.1f}s | "
        f"valid={stats['ok']}, http_err={stats['http_err']}, short/empty={stats['short_or_empty']}, "
        f"timeout/conn={stats['timeout_or_conn']}, non_html={stats['non_html']}"
    )
    if valid:
        niche_counts = pd.Series([r["niche"] for r in valid]).value_counts().to_dict()
        log(f"  -> Extracted pages per niche: {niche_counts}")
    return pd.DataFrame(valid)


db = get_con()
if os.path.exists(BUFFER_PATH):
    log(f"[3.1] Found parquet buffer at {BUFFER_PATH}; loading unseen rows...")
    df = pd.read_parquet(BUFFER_PATH)
    df["source_url"] = df["source_url"].map(normalize_url)
    df = df[df["source_url"] != ""]
    df["scraped_at"] = pd.to_datetime(df["scraped_at"], utc=True)
    df = df.drop_duplicates("source_url")
    seen_set = {u for (u,) in db.execute("SELECT source_url FROM seen_urls").fetchall()}
    df = df[~df["source_url"].isin(seen_set)]
    attempted = df["source_url"].tolist()
    log(f"  -> Loaded {len(df)} unseen rows from parquet buffer.")
else:
    targets = filter_unseen(discover_urls())
    attempted = [u for _, u in targets]
    df = run_async(scrape(targets)) if targets else pd.DataFrame()


def mark_seen(urls):
    if not urls:
        return
    db = get_con()
    now = datetime.now(timezone.utc)
    unique_urls = list(dict.fromkeys(urls))
    seen_df = pd.DataFrame({"source_url": unique_urls, "first_seen": now})
    db.register("seen_df_view", seen_df)
    db.execute(
        "INSERT OR IGNORE INTO seen_urls (source_url, first_seen) "
        "SELECT source_url, first_seen FROM seen_df_view"
    )
    db.unregister("seen_df_view")
    log(f"  -> Logged {len(unique_urls)} attempted URLs into 'minedb.seen_urls'.")


def apply_retention():
    db = get_con()
    cutoff = f"CAST(NOW() AS TIMESTAMPTZ) - INTERVAL '{RETENTION_DAYS} days'"
    db.execute(f"DELETE FROM seen_urls WHERE first_seen < {cutoff}")
    for n in NICHE_QUERIES:
        db.execute(f"DELETE FROM {n} WHERE scraped_at < {cutoff}")
    log(f"  -> Applied {RETENTION_DAYS}-day rolling retention policy across all tables.")


if df.empty:
    log("[3.4] Zero valid new pages extracted in this cycle. Marking URLs seen & exiting.")
    mark_seen(attempted)
    apply_retention()
    get_con().close()
    save_and_shutdown_session()

In [ ]:
# ----------------------------- PII SCRUB (All niches for PDPL safety) --------
log(f"[4.1] Running PDPL PII scrubber (masking emails & phone numbers) across {len(df)} pages...")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE_RE = re.compile(r"(?<!\d)(?:\+?\d[\d\s().-]{7,}\d)(?!\d)")
ARABIC_CHAR_RE = re.compile(r"[\u0600-\u06FF]")

pii_emails_masked = 0
pii_phones_masked = 0

def scrub_pii(text: str) -> str:
    global pii_emails_masked, pii_phones_masked
    text, n_e = EMAIL_RE.subn("[EMAIL]", text)
    text, n_p = PHONE_RE.subn("[PHONE]", text)
    pii_emails_masked += n_e
    pii_phones_masked += n_p
    return text

df["raw_text"] = df["raw_text"].map(scrub_pii)
log(f"  -> PII scrub complete: masked {pii_emails_masked} emails and {pii_phones_masked} phone numbers.")

# ----------------------------- LLM QUALITY GATE -----------------------------
import gc
log("[4.2] Clearing residual GPU memory before loading LLM...")
if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    alloc_mb = torch.cuda.memory_allocated(0) / (1024 ** 2)
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / (1024 ** 3)
    log(f"  -> GPU 0 pre-load state: allocated={alloc_mb:.1f} MB | free={free_gb:.2f} GB")

t_load = time.time()
log(f"[4.3] Loading {MODEL_ID} (FP16) onto GPU 0 from {CACHED_MODEL_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(CACHED_MODEL_DIR, padding_side="left", local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CACHED_MODEL_DIR,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    local_files_only=True,
)
model.eval()
vram_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
log(f"  -> Model loaded on {model.device} in {time.time() - t_load:.1f}s | GPU 0 VRAM allocated: {vram_gb:.2f} GB")


def build_prompt(niche: str, text: str) -> str:
    user = f"Niche: {niche}\nSchema for extracted_data: {NICHE_SCHEMA[niche]}\n\nTEXT:\n{text}"
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


NULL_STRINGS = {"unknown", "n/a", "na", "none", "null", "not specified", "not available", ""}


def clean_extracted_dict(d: dict) -> dict:
    """Normalize placeholder null strings into real JSON None and ensure at least 1 meaningful field."""
    cleaned = {}
    for k, v in d.items():
        if isinstance(v, str) and v.strip().lower() in NULL_STRINGS:
            cleaned[k] = None
        elif isinstance(v, list):
            cleaned[k] = [x for x in v if not (isinstance(x, str) and x.strip().lower() in NULL_STRINGS)]
        else:
            cleaned[k] = v
    return cleaned


def parse_reply(niche: str, reply: str):
    """Returns (parsed_obj_or_None, reason_tag)."""
    cleaned_reply = re.sub(r"^```(?:json)?\s*|\s*```$", "", reply.strip(), flags=re.IGNORECASE)
    s, e = cleaned_reply.find("{"), cleaned_reply.rfind("}") + 1
    if s == -1 or e <= s:
        return None, "invalid_json_bounds"
    try:
        obj = json.loads(cleaned_reply[s:e])
    except Exception:
        return None, "json_decode_error"

    if obj.get("is_sellable") is not True:
        return None, f"not_sellable(score={obj.get('value_score', '?')})"

    score = int(obj.get("value_score", 0))
    if score < MIN_VALUE_SCORE:
        return None, f"low_score({score}<{MIN_VALUE_SCORE})"

    if not isinstance(obj.get("extracted_data"), dict) or not obj["extracted_data"]:
        return None, "empty_schema"

    cleaned = clean_extracted_dict(obj["extracted_data"])

    # Enforce genuine Arabic script in arabic_nlp records so English marketing copy doesn't slip through
    if niche == "arabic_nlp":
        ar_txt = cleaned.get("clean_arabic_text") or ""
        ar_chars = len(ARABIC_CHAR_RE.findall(str(ar_txt)))
        if ar_chars < 25:
            return None, f"non_arabic_text(ar_chars={ar_chars})"

    non_null_fields = sum(1 for v in cleaned.values() if v not in (None, [], ""))
    if non_null_fields < 1:
        return None, "all_fields_null"

    obj["extracted_data"] = cleaned
    return obj, f"KEEP(score={score}, fields={non_null_fields})"


log(f"[5.1] Formatting & length-sorting {len(df)} prompts for batched inference (batch_size={LLM_BATCH_SIZE})...")
df = df.assign(prompt=[build_prompt(n, t) for n, t in zip(df["niche"], df["raw_text"])])
df = df.assign(plen=df["prompt"].str.len()).sort_values("plen")
rows = list(df.itertuples())
sellable = []
discard_reasons = {}
total_batches = (len(rows) + LLM_BATCH_SIZE - 1) // LLM_BATCH_SIZE
t_inf = time.time()

for b_idx, i in enumerate(range(0, len(rows), LLM_BATCH_SIZE), start=1):
    t_b = time.time()
    batch = rows[i:i + LLM_BATCH_SIZE]
    inputs = tokenizer(
        [r.prompt for r in batch],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1536,
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

    for r, reply in zip(batch, decoded):
        obj, reason = parse_reply(r.niche, reply.strip())
        short_url = (r.source_url[:62] + "...") if len(r.source_url) > 65 else r.source_url
        if obj:
            sellable.append({
                "niche": r.niche,
                "source_url": r.source_url,
                "scraped_at": r.scraped_at,
                "value_score": int(obj["value_score"]),
                "cleaned_json": json.dumps(obj["extracted_data"], ensure_ascii=False),
            })
            log(f"    + [{r.niche}] {reason} | {short_url}")
        else:
            tag = reason.split("(")[0]
            discard_reasons[tag] = discard_reasons.get(tag, 0) + 1
            log(f"    - [{r.niche}] DROP:{reason} | {short_url}")

    del inputs, out, new_tokens
    torch.cuda.empty_cache()
    peak_gb = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
    done = min(i + LLM_BATCH_SIZE, len(rows))
    log(f"  -> Batch [{b_idx}/{total_batches}] ({done}/{len(rows)} pages) done in {time.time() - t_b:.1f}s | kept={len(sellable)} | peak_vram={peak_gb:.2f} GB")

log(f"[5.2] Quality gate finished in {time.time() - t_inf:.1f}s: {len(rows)} evaluated -> {len(sellable)} KEPT, {len(rows) - len(sellable)} DISCARDED.")
if discard_reasons:
    log(f"  -> Discard breakdown: {discard_reasons}")

In [ ]:
# ----------------------------- SAVE -----------------------------------------
log("[6.1] Marking attempted URLs in MotherDuck 'seen_urls'...")
db = get_con()
mark_seen(attempted)

if sellable:
    log(f"[6.2] Inserting {len(sellable)} sellable records into MotherDuck niche tables...")
    clean_df = pd.DataFrame(sellable).drop_duplicates(["niche", "source_url"])
    for niche, group in clean_df.groupby("niche"):
        existing_niche_urls = {
            r[0] for r in db.execute(f"SELECT source_url FROM {niche}").fetchall()
        }
        group = group[~group["source_url"].isin(existing_niche_urls)]
        if group.empty:
            log(f"  -> 'minedb.{niche}': 0 new rows (all already existed).")
            continue
        db.register("group_view", group)
        db.execute(f"""
            INSERT INTO {niche} (niche, source_url, scraped_at, value_score, cleaned_json)
            SELECT niche, source_url, scraped_at, value_score, cleaned_json FROM group_view
        """)
        db.unregister("group_view")
        new_total = db.execute(f"SELECT COUNT(*) FROM {niche}").fetchone()[0]
        log(f"  -> 'minedb.{niche}': inserted +{len(group)} new rows (table total now {new_total}).")
    print("\n--- SAMPLE EXTRACTED RECORDS ---", flush=True)
    print(clean_df[["niche", "value_score", "cleaned_json"]].head(8).to_string(index=False), flush=True)
else:
    log(f"[6.2] No rows met value_score >= {MIN_VALUE_SCORE}. 'minedb' tables remain clean.")

log("[6.3] Running rolling retention cleanup & closing MotherDuck connection...")
apply_retention()
db.close()
log("=== PIPELINE COMPLETED SUCCESSFULLY ===")

save_and_shutdown_session()